In [ ]:
# Только для разработки: открыть раздел можно из notebooks/ или корня проекта
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'config/build/project.json').is_file()
)
_notebook_os.chdir(_notebook_root)


# Модель классификации для задачи оценки риска


### 0 Предварительный анализ

Цель этапа — уточнить семантику столбцов, чтобы корректно поставить задачу

Исходные описания столбцов короткие: в них не указаны единицы измерения и значения категорий, не везде понятно, как читать числа

Поэтому загружаем данные, определяем типы и домены значений и на их основе интерпретируем каждый столбец


#### 0.1 Исходные описания и загрузка данных

От автора датасета есть только короткие англоязычные описания столбцов (с Kaggle)


In [1]:
import pandas as pd

# убираем незначащие нули
pd.set_option('display.float_format',
              lambda x: f'{x:.6f}'.rstrip('0').rstrip('.'))
# не обрезаем длинный текст в ячейках таблиц
pd.set_option('display.max_colwidth', None)


def fmt_number(x):
    return f'{x:.6f}'.rstrip('0').rstrip('.')


# описания столбцов от автора датасета
source_descriptions = {
    'person_age': 'Age',
    'person_income': 'Annual Income',
    'person_home_ownership': 'Home ownership',
    'person_emp_length': 'Employment length (in years)',
    'loan_intent': 'Loan intent',
    'loan_grade': 'Loan grade',
    'loan_amnt': 'Loan amount',
    'loan_int_rate': 'Interest rate',
    'loan_status': 'Loan status (0 is non default 1 is default)',
    'loan_percent_income': 'Percent income',
    'cb_person_default_on_file': 'Historical default',
    'cb_person_cred_hist_length': 'Credit history length',
}

data_raw = pd.read_csv('./dataset/credit_risk_dataset.csv')
data = data_raw.copy()

display(pd.Series(source_descriptions, name='Описание из источника').to_frame())

,Описание из источника
person_age,Age
person_income,Annual Income
person_home_ownership,Home ownership
person_emp_length,Employment length (in years)
loan_intent,Loan intent
loan_grade,Loan grade
loan_amnt,Loan amount
loan_int_rate,Interest rate
loan_status,Loan status (0 is non default 1 is default)
loan_percent_income,Percent income


#### 0.2 Типы и домены значений

Для каждого столбца определяем по данным фактический тип, число различных значений и домен: диапазон [min, max] для числовых, множество значений для категориальных


In [2]:
# столбец считаем категориальным, если он нечисловой или у него мало различных значений
MAX_CATEGORIES = 10

def is_categorical(column):
    values = data[column]
    return not pd.api.types.is_numeric_dtype(values) or values.nunique() <= MAX_CATEGORIES


def domain(column):
    values = data[column].dropna()
    if is_categorical(column):
        return '{' + ', '.join(str(v) for v in sorted(values.unique())) + '}'
    return f'[{fmt_number(values.min())}, {fmt_number(values.max())}]'


columns_info = pd.DataFrame(index=data.columns)
columns_info['Описание из источника'] = columns_info.index.map(source_descriptions)
columns_info['Тип в данных'] = data.dtypes.astype(str)
columns_info['Домен'] = columns_info.index.map(domain)

categorical = columns_info.index.map(is_categorical)
display(columns_info[~categorical].style.set_caption('Числовые столбцы'))
display(columns_info[categorical].style.set_caption('Категориальные столбцы'))

,Описание из источника,Тип в данных,Домен
person_age,Age,int64,"[20, 144]"
person_income,Annual Income,int64,"[4000, 6000000]"
person_emp_length,Employment length (in years),float64,"[0, 123]"
loan_amnt,Loan amount,int64,"[500, 35000]"
loan_int_rate,Interest rate,float64,"[5.42, 23.22]"
loan_percent_income,Percent income,float64,"[0, 0.83]"
cb_person_cred_hist_length,Credit history length,int64,"[2, 30]"


,Описание из источника,Тип в данных,Домен
person_home_ownership,Home ownership,str,"{MORTGAGE, OTHER, OWN, RENT}"
loan_intent,Loan intent,str,"{DEBTCONSOLIDATION, EDUCATION, HOMEIMPROVEMENT, MEDICAL, PERSONAL, VENTURE}"
loan_grade,Loan grade,str,"{A, B, C, D, E, F, G}"
loan_status,Loan status (0 is non default 1 is default),int64,"{0, 1}"
cb_person_default_on_file,Historical default,str,"{N, Y}"


#### 0.3 Интерпретация столбцов

На основании описания из источника и доменов значений делаем развёрнутые описания столбцов и определяем типы категориальных признаков

_Дефолт — неисполнение заёмщиком обязательств по кредиту: прекращение платежей или существенная просрочка_


In [3]:
# порядок категорий из данных не выводится — задаём вручную
ORDINAL_COLUMNS = {'loan_grade'}

def semantic_type(column):
    if not is_categorical(column):
        return 'числовой'
    if data[column].nunique() == 2:
        return 'бинарный'
    if column in ORDINAL_COLUMNS:
        return 'порядковый'
    return 'номинальный'


# уточнённые описания: единицы измерения, расшифровка значений
interpretations = {
    'person_age': 'возраст заёмщика, лет',
    'person_income': 'годовой доход',
    'person_home_ownership': 'жильё:\n'
                             '• RENT — снимает\n'
                             '• OWN — своё\n'
                             '• MORTGAGE — в ипотеке\n'
                             '• OTHER — другое',
    'person_emp_length': 'стаж на текущей работе, лет',
    'loan_intent': 'цель кредита:\n'
                   '• PERSONAL — личные нужды\n'
                   '• EDUCATION — образование\n'
                   '• MEDICAL — лечение\n'
                   '• VENTURE — бизнес\n'
                   '• HOMEIMPROVEMENT — ремонт жилья\n'
                   '• DEBTCONSOLIDATION — рефинансирование других долгов',
    'loan_grade': 'грейд займа от A (надёжный) до G (рискованный)',
    'loan_amnt': 'сумма кредита',
    'loan_int_rate': 'процентная ставка, % годовых',
    'loan_status': 'статус кредита:\n'
                   '• 0 — платит\n'
                   '• 1 — дефолт',
    'loan_percent_income': 'доля кредита от годового дохода',
    'cb_person_default_on_file': 'были ли дефолты раньше (по данным кредитного бюро):\n'
                                 '• Y — были\n'
                                 '• N — не было',
    'cb_person_cred_hist_length': 'длина кредитной истории, лет',
}

columns_info['Фактический тип'] = columns_info.index.map(semantic_type)
columns_info['Описание'] = columns_info.index.map(interpretations)
display(columns_info[['Описание из источника', 'Описание', 'Фактический тип']]
        .style.set_properties(**{'white-space': 'pre-line', 'text-align': 'left'}))

,Описание из источника,Описание,Фактический тип
person_age,Age,"возраст заёмщика, лет",числовой
person_income,Annual Income,годовой доход,числовой
person_home_ownership,Home ownership,жильё: • RENT — снимает • OWN — своё • MORTGAGE — в ипотеке • OTHER — другое,номинальный
person_emp_length,Employment length (in years),"стаж на текущей работе, лет",числовой
loan_intent,Loan intent,цель кредита: • PERSONAL — личные нужды • EDUCATION — образование • MEDICAL — лечение • VENTURE — бизнес • HOMEIMPROVEMENT — ремонт жилья • DEBTCONSOLIDATION — рефинансирование других долгов,номинальный
loan_grade,Loan grade,грейд займа от A (надёжный) до G (рискованный),порядковый
loan_amnt,Loan amount,сумма кредита,числовой
loan_int_rate,Interest rate,"процентная ставка, % годовых",числовой
loan_status,Loan status (0 is non default 1 is default),статус кредита: • 0 — платит • 1 — дефолт,бинарный
loan_percent_income,Percent income,доля кредита от годового дохода,числовой
